In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

print("Path to dataset files:", path)

100%|██████████| 786M/786M [00:07<00:00, 116MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nodoubttome/skin-cancer9-classesisic/versions/1


In [2]:
# Cell 1: Install Dependencies
!pip install -q timm thop scikit-learn xgboost torchinfo

import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import timm
from thop import profile
from torchinfo import summary
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from xgboost import XGBClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [14]:
# Cell 2: Load Real ISIC Dataset from kagglehub Path
import os
import kagglehub
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms

# Download dataset
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Path to dataset files:", path)

# Define standard image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets using ImageFolder, pointing to the 'Train' and 'Test' subdirectories
base_data_path = os.path.join(path, 'Skin cancer ISIC The International Skin Imaging Collaboration')

train_dataset = ImageFolder(root=os.path.join(base_data_path, 'Train'), transform=transform)
test_dataset = ImageFolder(root=os.path.join(base_data_path, 'Test'), transform=transform)

print(f"Total training images found: {len(train_dataset)}")
print(f"Total testing images found: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic
Total training images found: 2239
Total testing images found: 118
Classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']


In [9]:
# Cell 3: Transfer Learning Training & Evaluation Loop (Tables 1 & 3)
model_names = ['resnet50', 'vgg16', 'resnet18', 'efficientnet_b0'] # Add others as required
results_table1 = []
results_table3 = []

for name in model_names:
    print(f"\n--- Training & Benchmarking: {name} ---")
    # Load model with timm, setting num_classes to 9
    model = timm.create_model(name, pretrained=True, num_classes=9).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    # 1. Compute Efficiency Metrics (Table 3 inputs)
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    try:
        macs, params = profile(model, inputs=(dummy_input,), verbose=False)
        flops_g = (macs * 2) / 1e9 # Convert MACs to FLOPs
    except:
        flops_g, params = 0.0, sum(p.numel() for p in model.parameters())

    param_m = params / 1e6

    # Model Size in MB
    torch.save(model.state_dict(), "temp.pth")
    import os
    model_size_mb = os.path.getsize("temp.pth") / (1024 * 1024)
    os.remove("temp.pth")

    # Quick 1-epoch training simulation
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        break # Just 1 batch for demonstration script speed

    # Evaluation & Inference Time (Table 1 & 3)
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    start_time = time.time()
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    total_time = time.time() - start_time
    inference_time_ms = (total_time / len(test_dataset)) * 1000

    acc = accuracy_score(all_labels, all_preds) * 100
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)

    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro') * 100
    except:
        auc = 0.0

    results_table1.append({
        "Model": name, "Accuracy (%)": f"{acc:.2f}", "Precision (%)": f"{precision*100:.2f}",
        "Recall (%)": f"{recall*100:.2f}", "F1-Score (%)": f"{f1*100:.2f}", "AUC (%)": f"{auc:.2f}"
    })

    results_table3.append({
        "Model": name, "Parameters (M)": f"{param_m:.2f}", "Model Size (MB)": f"{model_size_mb:.2f}",
        "FLOPs (G)": f"{flops_g:.2f}", "Inference Time (ms)": f"{inference_time_ms:.2f}", "Accuracy (%)": f"{acc:.2f}"
    })

print("\nPipeline Complete!")


--- Training & Benchmarking: resnet50 ---

--- Training & Benchmarking: vgg16 ---

--- Training & Benchmarking: resnet18 ---


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            


--- Training & Benchmarking: efficientnet_b0 ---


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            


Pipeline Complete!


In [15]:
# Cell 4: Deep Feature Extraction & Classical Classifiers (Table 2)
# Using a feature extractor backbone (e.g., ResNet18 without final classification head)
backbone = timm.create_model('resnet18', pretrained=True, num_classes=0).to(device)
backbone.eval()

def extract_features(loader):
    features, targets = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            feats = backbone(images)
            features.append(feats.cpu().numpy())
            targets.append(labels.numpy())
    return np.vstack(features), np.concatenate(targets)

X_train, y_train = extract_features(train_loader)
X_test, y_test = extract_features(test_loader)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(),
    "Linear SVM": LinearSVC(max_iter=1000),
    "RBF-SVM": SVC(probability=True),
    "XGBoost": XGBClassifier()
}

table2_results = []
for clf_name, clf in classifiers.items():
    print(f"Training Classifier: {clf_name}...")
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)

    table2_results.append({
        "Classifier": clf_name, "Accuracy (%)": f"{acc:.2f}",
        "Precision (%)": f"{prec*100:.2f}", "Recall (%)": f"{rec*100:.2f}",
        "F1-Score (%)": f"{f1*100:.2f}"
    })

Training Classifier: Logistic Regression...
Training Classifier: Decision Tree...
Training Classifier: Random Forest...
Training Classifier: K-Nearest Neighbors (KNN)...
Training Classifier: Linear SVM...
Training Classifier: RBF-SVM...
Training Classifier: XGBoost...


In [16]:
import pandas as pd

In [17]:
df_table1 = pd.DataFrame(results_table1)
display(df_table1)

,Model,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
0,resnet50,1.27,11.11,0.14,0.28,0.00
1,vgg16,100.00,100.00,100.00,100.00,0.00
2,resnet18,30.72,12.50,3.84,5.88,0.00
3,efficientnet_b0,4.03,11.11,0.45,0.86,0.00


In [18]:
df_table2 = pd.DataFrame(table2_results)
display(df_table2)

,Classifier,Accuracy (%),Precision (%),Recall (%),F1-Score (%)
0,Logistic Regression,50.00,47.74,50.00,44.43
1,Decision Tree,23.73,18.60,25.46,18.68
2,Random Forest,31.36,32.67,34.72,27.16
3,K-Nearest Neighbors (KNN),28.81,32.71,29.63,27.93
4,Linear SVM,45.76,45.09,46.53,42.63
5,RBF-SVM,49.15,52.82,49.31,44.43
6,XGBoost,42.37,40.13,43.75,37.51


In [19]:
df_table3 = pd.DataFrame(results_table3)
display(df_table3)

,Model,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
0,resnet50,23.53,90.06,8.26,217.90,1.27
1,vgg16,134.30,512.32,30.93,646.03,100.00
2,resnet18,11.18,42.74,3.65,109.90,30.72
3,efficientnet_b0,3.98,15.73,0.77,88.33,4.03
